GroupDNA: Whatsapp Chat Analysier

In [223]:
# Import Libraries

import numpy as np
from datetime import datetime

## 1.Chat Parser
Convert raw chat lines into structured records

In [224]:
def chat_parser(file_path):
    messages = []
    media_count = 0
    deleted_count = 0
    system_count = 0

    with open(file_path, encoding="utf-8") as f:
        lines = f.readlines()

    previous_message = None

    for line in lines:
        line = line.strip()
        if line == "":
            continue

        # Handle multiline messages safely
        if len(line) < 8 or line[2] != "/" or line[5] != "/":
            if previous_message is not None:
                # Append text key safely using your data model name
                previous_message["message"] += " " + line
            continue

        try:
            timestamp, remaining = line.split(" - ", 1)
        except:
            continue

        if ":" not in remaining:
            system_count += 1
            continue

        sender, message = remaining.split(":", 1)
        message = message.strip()

        data = {
            "timestamp": timestamp.strip(),
            "sender": sender.strip(),
            "message": message
        }

        if "<Media omitted>" in message:
            media_count += 1
        elif "This message was deleted" in message:
            deleted_count += 1

        messages.append(data)
        previous_message = data

    return messages, system_count, media_count, deleted_count

**2. Group Overview**

In [225]:
def group_overview(messages):
    print("=" * 60)
    print("                 GROUP OVERVIEW")
    print("=" * 60)

    participants = set()
    message_count = {}

    first_date = messages[0]["timestamp"].split(",")[0]
    last_date = messages[-1]["timestamp"].split(",")[0]

    start_date = datetime.strptime(first_date, "%d/%m/%y")
    end_date = datetime.strptime(last_date, "%d/%m/%y")
    total_days = (end_date - start_date).days + 1

    for msg in messages:
        sender = msg["sender"]
        participants.add(sender)
        message_count[sender] = message_count.get(sender, 0) + 1

    sorted_messages = sorted(message_count.items(), key=lambda x: x[1], reverse=True)

    # Add text-based bar chart

    print()
    print(f"{'Participant':<15} | {'Messages Sent':<15}")
    print()

    if sorted_messages:
        max_messages_overall = sorted_messages[0][1]
        for sender, count in sorted_messages:
            bar_length = int(np.ceil((count / max_messages_overall) * 20)) # Scale bar to 20 characters
            bar = "█" * bar_length
            print(f"{sender:<15} {bar:<20} {count}")

**3. To Analyis Most Active Day and Most Active Hour**

In [226]:
def most_active_day_hour(messages):

  #print("=" * 60)
  #print("          MOST ACTIVE DAY & HOUR")
  #print("=" * 60)

  day_count ={}
  hour_count = {}

  #To Calculate messages for each day and hour
  for msg in messages:

    timestamp = msg["timestamp"]

    date = timestamp.split(",")[0]
    time = timestamp.split(",")[1].strip()

    hour = int(time.split(":")[0])

    #Count day
    if date in day_count:
      day_count[date] += 1
    else:
      day_count[date] = 1

    #Count hour
    if hour in hour_count:
      hour_count[hour] += 1
    else:
      hour_count[hour] = 1

  #Find Busiest day
  max_day = ""
  max_day_messages = 0

  for day in day_count:
    if day_count[day] > max_day_messages:
      max_day_messages = day_count[day]
      max_day = day

  #Find busiest hour
  max_hour = 0
  max_hour_messages = 0

  for hour in hour_count:
    if hour_count[hour] > max_hour_messages:
      max_hour_messages = hour_count[hour]
      max_hour = hour

  #To convert data into readabl formate
  day_object = datetime.strptime(max_day,"%d/%m/%y")

  print("Busiest Day  :", day_object.strftime("%d %B %Y"),
        f"({max_day_messages} messages)")

  print("Busiest Hour :", f"{max_hour:02d}:00 - {(max_hour+1)%24:02d}:00",
        f"({max_hour_messages} messages)")

**4. Activity Heatmap Function**

In [227]:
def activity_heatmap(messages):
    print("=" * 75)
    print("                 ACTIVITY HEATMAP (Messages by Hour)")
    print("=" * 75)

    participants = []
    for msg in messages:
        if msg["sender"] not in participants:
            participants.append(msg["sender"])
    participants.sort()

    # Create Matrix
    heatmap = np.zeros((len(participants), 24), dtype=int)

    # Fill Matrix completely BEFORE printing
    for msg in messages:
        sender = msg["sender"]
        time = msg["timestamp"].split(",")[1].strip()
        hour = int(time.split(":")[0])
        row = participants.index(sender)
        heatmap[row][hour] += 1

    # Print Hour Header (Moved completely outside the loop)
    print("         ", end="")
    for hour in range(24):
        print(f"{hour:02d}", end=" ")
    print("\n")

    # Print Heatmap Rows
    for i in range(len(participants)):
        print(f"{participants[i]:<8}", end=" ")
        maximum = max(heatmap[i])

        for value in heatmap[i]:
            if value == 0:
                symbol = ". "
            else:
                percentage = value / maximum if maximum > 0 else 0
                if percentage <= 0.25:
                    symbol = "░ "
                elif percentage <= 0.50:
                    symbol = "▒ "
                else:
                    symbol = "█ "
            print(symbol, end="")
        print()

    return heatmap

**5. Top Word Analysis**

In [228]:
def top_words(messages):

    print("\n")
    print("=" * 60)
    print("        THIS GROUP'S FAVOURITE WORDS")
    print("=" * 60)

    stop_words = [
        "the","is","am","are","was","were","to","of","for","in",
        "on","at","and","or","a","an","i","me","my","you","your",
        "he","she","it","we","they","this","that","these","those",
        "ok","okay","yes","no","ha","haha","lol","hmm","hmmm"
    ]

    word_count = {}

    for msg in messages:

        text = msg["message"].lower()

        # Skip media & deleted messages
        if text == "<media omitted>" or text == "this message was deleted":
            continue

        words = text.split()

        for word in words:

            word = word.strip(".,!?()[]{}:;'\"")

            if word == "":
                continue

            if word in stop_words:
                continue

            if word in word_count:
                word_count[word] += 1
            else:
                word_count[word] = 1

    # Sort words
    sorted_words = sorted(word_count.items(),
                          key=lambda x: x[1],
                          reverse=True)

    print()

    # Find largest frequency
    maximum = sorted_words[0][1]

    # Show Top 10 Words
    for i in range(10):

        word = sorted_words[i][0]
        count = sorted_words[i][1]

        # Scale bar to maximum 25 blocks
        bar_length = int((count / maximum) * 25)

        bar = "█" * bar_length

        print(f"{word:<12} {bar:<25} {count}")

**6. To Find Response Speed & Silent Streak**

In [229]:
from datetime import datetime

def response_speed(messages):

    #print("=" * 60)
    #print("          RESPONSE SPEED & SILENT STREAK")
    #print("=" * 60)

    total_response_time = 0
    response_count = 0

    longest_gap = 0

    previous_sender = messages[0]["sender"]

    previous_time = datetime.strptime(
        messages[0]["timestamp"],
        "%d/%m/%y, %H:%M"
    )

    for i in range(1, len(messages)):

        current_sender = messages[i]["sender"]

        current_time = datetime.strptime(
            messages[i]["timestamp"],
            "%d/%m/%y, %H:%M"
        )

        difference = (current_time - previous_time).total_seconds() / 60

        # Longest silent streak
        if difference > longest_gap:
            longest_gap = difference

        # Response time (only when sender changes)
        if current_sender != previous_sender:

            total_response_time += difference
            response_count += 1

        previous_sender = current_sender
        previous_time = current_time

    if response_count > 0:
        average_response = total_response_time / response_count
    else:
        average_response = 0

    # Convert longest gap into days, hours, minutes
    days = int(longest_gap // (24 * 60))
    hours = int((longest_gap % (24 * 60)) // 60)
    minutes = int(longest_gap % 60)

    print("Average Response Time : {:.2f} minutes".format(average_response))

    print("Longest Silent Streak :", end=" ")

    if days > 0:
        print(days, "days", hours, "hours", minutes, "minutes")
    elif hours > 0:
        print(hours, "hours", minutes, "minutes")
    else:
        print(minutes, "minutes")

**Feature 7 – Personality Archetypes**

In [230]:
from datetime import datetime

def personality_report(messages):


    # Message Count

    message_count = {}

    # Hour Activity
    hour_count = {}

    for msg in messages:

        sender = msg["sender"]
        # Corrected: Access 'timestamp' key and extract the hour
        hour = int(msg["timestamp"].split(', ')[1].split(":")[0])

        # Count messages
        if sender in message_count:
            message_count[sender] += 1
        else:
            message_count[sender] = 1

        # Count hours
        if sender not in hour_count:
            hour_count[sender] = [0] * 24

        hour_count[sender][hour] += 1

    # Average messages
    average_messages = np.mean(list(message_count.values()))

    print("=" * 60)
    print("          PERSONALITY ARCHETYPES")
    print("=" * 60)

    for person in sorted(message_count):

        total = message_count[person]

        busiest_hour = hour_count[person].index(max(hour_count[person]))


        # Decide personality

        if total > average_messages * 1.5:

            personality = "Spammer"

        elif total < average_messages * 0.4:

            personality = "Ghost"

        elif busiest_hour >= 22 or busiest_hour <= 4:

            personality = "Night Owl"

        elif busiest_hour >= 6 and busiest_hour <= 10:

            personality = "Early Bird"

        elif total > average_messages:

            personality = "Storyteller"

        else:

            personality = "Caretaker"

        print(f"{person:<12} --> {personality}")

**Final Report**

In [231]:
# ============================
#           MAIN PROGRAM
# ============================

# Define group_name (can be set by user, using a placeholder for now)
group_name = "Hostel Bois 4ever"

file_path = "hostel_bois.txt"
messages, system_count, media_count, deleted_count = chat_parser(file_path)

# Calculate variables needed for the header after messages are parsed
total_messages = len(messages)

# Calculate start_date, end_date, and total_days
if messages:
    first_date_str = messages[0]["timestamp"].split(",")[0]
    last_date_str = messages[-1]["timestamp"].split(",")[0]
    start_date = datetime.strptime(first_date_str, "%d/%m/%y")
    end_date = datetime.strptime(last_date_str, "%d/%m/%y")
    total_days = (end_date - start_date).days + 1
else:
    start_date = None
    end_date = None
    total_days = 0

# Calculate sender_counts for the header
sender_counts = {}
for msg in messages:
    sender = msg["sender"]
    sender_counts[sender] = sender_counts.get(sender, 0) + 1


print("======================================================================")
print(f'   GROUP NAME  -  "{group_name}"')
print(f"   {total_days} days   •   {total_messages:,} messages   •   {len(sender_counts)} members")
print("======================================================================")
print()
print(f"Period          : {start_date.strftime('%d %B %y')} to {end_date.strftime('%d %B %y')}")

print("\nChat Parsed Successfully!")
print("Total Parsed Messages :", len(messages))

print("\n\n--- FEATURE 1: PARSER SUMMARY ---")
print(f"System Alerts/Actions  : {system_count}")
print(f"Media Files Shared     : {media_count}")
print(f"Deleted Messages       : {deleted_count}")

print("\n\n--- FEATURE 2 ---")
group_overview(messages)

print("\n\n--- FEATURE 3 ---")
most_active_day_hour(messages)

print("\n\n--- FEATURE 4 ---")
heatmap = activity_heatmap(messages)

print("\n\n--- FEATURE 5 ---")
top_words(messages)

print("\n\n--- FEATURE 6 ---")
response_speed(messages)

print("\n\n--- FEATURE 7 ---")
personality_report(messages)

print("=" * 80)
print("        Generated by GroupDNA • Built with Python + NumPy")
print("=" * 80)

   GROUP NAME  -  "Hostel Bois 4ever"
   60 days   •   3,174 messages   •   6 members

Period          : 01 April 24 to 30 May 24

Chat Parsed Successfully!
Total Parsed Messages : 3174


--- FEATURE 1: PARSER SUMMARY ---
System Alerts/Actions  : 4
Media Files Shared     : 32
Deleted Messages       : 15


--- FEATURE 2 ---
                 GROUP OVERVIEW

Participant     | Messages Sent  

Rahul           ████████████████████ 953
Priya           ████████████████     718
Neha            ██████████████       635
Aman            ███████████          490
Karan           ████████             354
Vikas           █                    24


--- FEATURE 3 ---
Busiest Day  : 04 May 2024 (76 messages)
Busiest Hour : 18:00 - 19:00 (248 messages)


--- FEATURE 4 ---
                 ACTIVITY HEATMAP (Messages by Hour)
         00 01 02 03 04 05 06 07 08 09 10 11 12 13 14 15 16 17 18 19 20 21 22 23 

Aman     █ █ █ █ █ . . . . . . . . . ░ ░ ░ ░ ░ ░ ░ ░ . █ 
Karan    . . . . . . . ░ ▒ ▒ █ ▒ █ █ █ █ █ 